# C5 — limpieza de los cuatro CSV reales

Los cuatro archivos tienen el mismo esquema, por lo que primero se conserva `_archivo_origen` y después pueden concatenarse de forma controlada.


## Principios de limpieza y trazabilidad — C5

Se agregan campos de control que permiten conocer el archivo y la fila de procedencia. Estos campos no sustituyen información original; sirven para rastrear cada registro durante todo el proceso.

En esta fuente se aplicarán las siguientes reglas:

- **Conservar el valor original.** Las columnas originales no se sobrescriben.
- **Crear columnas limpias separadas.** Una transformación produce una nueva columna identificable.
- **No asumir equivalencias.** Que dos valores, campos o registros se repitan no significa que representen exactamente lo mismo.
- **No eliminar folios por repetición.** Un folio repetido se marca para revisión, pero no se descarta automáticamente.
- **No inventar faltantes.** Un dato vacío continúa como faltante salvo que exista una regla documentada de la fuente.
- **Separar únicamente copias exactas cuando corresponda.** Se conserva la primera aparición y las copias exactas adicionales se envían a una tabla de excluidos.
- **Documentar qué se conserva y qué se separa.**
- **Registrar cada decisión en una bitácora.**


## 0. Cargar datos

En esta sección se carga `CARPETA_DATOS.zip` y se prepara el entorno de trabajo. Todavía no se modifica ningún registro; únicamente se localizan los archivos necesarios y se crean las funciones que se utilizarán durante la limpieza.


In [ ]:
from google.colab import files
from pathlib import Path
import pandas as pd
import numpy as np
import zipfile, unicodedata, re

subidos = files.upload()
nombre_zip = next(iter(subidos))

with zipfile.ZipFile(nombre_zip) as z:
    z.extractall("/content")

ROOT = Path("/content/CARPETA_DATOS")

def leer_csv(ruta):
    for enc in ["utf-8-sig","utf-8","latin-1","cp1252"]:
        for sep in [",",";"]:
            try:
                d = pd.read_csv(ruta, encoding=enc, sep=sep, low_memory=False)
                if len(d.columns) > 1:
                    return d
            except Exception:
                pass
    raise ValueError(f"No se pudo leer: {ruta}")

def texto_limpio(s):
    return (s.astype("string").str.strip().str.replace(r"\s+"," ",regex=True)
            .replace({"":pd.NA,"nan":pd.NA,"None":pd.NA,"NULL":pd.NA,"null":pd.NA}))

def normalizar(v):
    if pd.isna(v): return pd.NA
    v=" ".join(str(v).upper().strip().split())
    return "".join(c for c in unicodedata.normalize("NFD",v)
                   if unicodedata.category(c)!="Mn")

ALCALDIAS = {
"ALVARO OBREGON":"Álvaro Obregón","AZCAPOTZALCO":"Azcapotzalco",
"BENITO JUAREZ":"Benito Juárez","COYOACAN":"Coyoacán",
"CUAJIMALPA DE MORELOS":"Cuajimalpa de Morelos","CUAUHTEMOC":"Cuauhtémoc",
"GUSTAVO A MADERO":"Gustavo A. Madero","IZTACALCO":"Iztacalco",
"IZTAPALAPA":"Iztapalapa","LA MAGDALENA CONTRERAS":"La Magdalena Contreras",
"MIGUEL HIDALGO":"Miguel Hidalgo","MILPA ALTA":"Milpa Alta",
"TLAHUAC":"Tláhuac","TLALPAN":"Tlalpan",
"VENUSTIANO CARRANZA":"Venustiano Carranza","XOCHIMILCO":"Xochimilco"}


## 1. Leer los cuatro CSV

Aquí se leen los archivos CSV correspondientes a la fuente. Cada archivo conserva su nombre y estructura original para poder identificar de dónde proviene cada registro antes de realizar cambios.


In [ ]:
C5_DIR=ROOT/"C5"
rutas_c5=sorted(C5_DIR.glob("*.csv"))
partes=[]
for p in rutas_c5:
    d=leer_csv(p)
    d.insert(0,"_archivo_origen",p.name)
    d.insert(1,"_fila_archivo",np.arange(2,len(d)+2))
    partes.append(d)
    print(p.name,d.shape)


## 2. Concatenar conservando procedencia

Los archivos que comparten la misma estructura se integran conservando su archivo de origen. La unión no significa que valores repetidos sean equivalentes ni provoca eliminaciones automáticas.


In [ ]:
c5=pd.concat(partes,ignore_index=True,sort=False)
c5.insert(0,"_fila_union",np.arange(1,len(c5)+1))
print(c5.shape)


## 3. Limpiar textos

Se crean versiones limpias de los campos de texto para corregir espacios y representaciones vacías. Las columnas originales permanecen intactas para poder comparar el valor recibido con el valor preparado.


In [ ]:
for c in ["folio","dia_semana","tipo_incidente_c4","incidente_c4","alcaldia_inicio",
          "codigo_cierre","clas_con_f_alarma","tipo_entrada","alcaldia_cierre",
          "alcaldia_catalogo","colonia_catalogo"]:
    if c in c5: c5[c+"_limpia"]=texto_limpio(c5[c])


## 4. Limpiar fecha de creación

La fecha se interpreta en una columna nueva. Si un valor no puede convertirse correctamente, se marca como problema de formato y se conserva el dato original; no se inventa una fecha.


In [ ]:
c5["fecha_creacion_limpia"]=pd.to_datetime(c5["fecha_creacion"],errors="coerce",dayfirst=True)
c5["_fecha_creacion_invalida"]=c5["fecha_creacion"].notna() & c5["fecha_creacion_limpia"].isna()


## 5. Limpiar fecha de cierre

La fecha se interpreta en una columna nueva. Si un valor no puede convertirse correctamente, se marca como problema de formato y se conserva el dato original; no se inventa una fecha.


In [ ]:
c5["fecha_cierre_limpia"]=pd.to_datetime(c5["fecha_cierre"],errors="coerce",dayfirst=True)
c5["_fecha_cierre_invalida"]=c5["fecha_cierre"].notna() & c5["fecha_cierre_limpia"].isna()


## 6. Limpiar hora de creación

La hora se transforma a un formato utilizable en una columna separada. Los valores originales se conservan y los formatos que no puedan interpretarse se marcan para revisión, sin eliminar la fila.


In [ ]:
c5["hora_creacion_limpia"]=pd.to_datetime(c5["hora_creacion"].astype("string").str.strip(),errors="coerce")
c5["hora"]=c5["hora_creacion_limpia"].dt.hour.astype("Int64")
c5["_hora_creacion_invalida"]=c5["hora_creacion"].notna() & c5["hora_creacion_limpia"].isna()


## 7. Crear franja horaria

A partir de las horas válidas se crea una franja horaria derivada. Esta variable permitirá analizar la frecuencia de hechos por horario sin reemplazar ni modificar la hora original.


In [ ]:
c5["franja_horaria"]=pd.cut(c5["hora"],[-1,5,11,17,23],
 labels=["Madrugada","Mañana","Tarde","Noche"])


## 8. Homologar alcaldía de catálogo

Se crea una versión homologada de la alcaldía para comparar los registros con las 16 alcaldías de CDMX. Los nombres que no puedan reconocerse se marcan, pero se conserva el texto original para revisión.


In [ ]:
c5["_alcaldia_normalizada"]=c5["alcaldia_catalogo"].apply(normalizar)
c5["alcaldia_limpia"]=c5["_alcaldia_normalizada"].map(ALCALDIAS).astype("string")
c5["_alcaldia_no_reconocida"]=c5["alcaldia_catalogo"].notna() & c5["alcaldia_limpia"].isna()


## 9. Limpiar longitud

Las coordenadas se convierten a formato numérico en columnas nuevas. Si un valor no puede interpretarse, permanece disponible en su columna original y se registra como un problema de calidad.


In [ ]:
c5["longitud_limpia"]=pd.to_numeric(c5["longitud"],errors="coerce")


## 10. Limpiar latitud

Las coordenadas se convierten a formato numérico en columnas nuevas. Si un valor no puede interpretarse, permanece disponible en su columna original y se registra como un problema de calidad.


In [ ]:
c5["latitud_limpia"]=pd.to_numeric(c5["latitud"],errors="coerce")


## 11. Duplicados exactos

Se buscan únicamente copias exactas de las filas. Se conserva una primera aparición y, cuando corresponde, las copias exactas adicionales se separan para evitar doble conteo. Una coincidencia parcial no se elimina.


In [ ]:
originales=["folio","fecha_creacion","hora_creacion","dia_semana","fecha_cierre","hora_cierre",
"tipo_incidente_c4","incidente_c4","alcaldia_inicio","codigo_cierre","clas_con_f_alarma",
"tipo_entrada","alcaldia_cierre","alcaldia_catalogo","colonia_catalogo","longitud","latitud"]
c5["_duplicado_exacto"]=c5.duplicated(originales,keep="first")


## 12. Folios repetidos

Se identifican los folios que aparecen más de una vez. Un folio repetido no se elimina automáticamente, porque puede corresponder a registros distintos relacionados con un mismo hecho. Solo se marca para revisión.


In [ ]:
c5["_folio_repetido"]=c5["folio"].notna() & c5.duplicated("folio",keep=False)


## 13. Exportar CSV consolidado y archivos limpios por periodo

Los resultados de la limpieza y las evidencias de auditoría se guardan como CSV independientes. Así se conserva por separado la base preparada, los controles de calidad, la conciliación y la bitácora.


In [ ]:
SALIDA=Path("/content/C5_RESULTADOS_CSV"); SALIDA.mkdir(exist_ok=True)
c5_limpio=c5.loc[~c5["_duplicado_exacto"]].copy()
c5_limpio.to_csv(SALIDA/"C5_consolidado_limpio.csv",index=False,encoding="utf-8-sig")

for origen,g in c5_limpio.groupby("_archivo_origen",dropna=False):
    nombre=Path(str(origen)).stem
    g.to_csv(SALIDA/f"{nombre}_limpio.csv",index=False,encoding="utf-8-sig")

for p in sorted(SALIDA.glob("*.csv")): print(p.name)


## 14. Descargar un CSV

En esta sección se carga `CARPETA_DATOS.zip` y se prepara el entorno de trabajo. Todavía no se modifica ningún registro; únicamente se localizan los archivos necesarios y se crean las funciones que se utilizarán durante la limpieza.


In [ ]:
archivos_csv=sorted(SALIDA.glob("*.csv"))
for i,p in enumerate(archivos_csv,1): print(i,p.name)
n=int(input("Número: "))
if 1<=n<=len(archivos_csv): files.download(str(archivos_csv[n-1]))


## Auditoría 1 — Vacíos por columna

Se cuenta cuántos valores faltantes existen en cada columna y qué porcentaje representan. Los vacíos no se sustituyen por valores inventados; se documentan para decidir después si afectan un análisis específico.

Este bloque cuenta los valores faltantes sin modificarlos. Sirve para documentar qué campos tienen ausencia de información.


In [ ]:
perfil_vacios = []
objetos = {}

# Detectar automáticamente las tablas principales disponibles en este notebook.
for nombre in ["tablas_ssc","c5","datos","datos_limpios","cdmx"]:
    if nombre in globals():
        objetos[nombre] = globals()[nombre]

if "tablas_ssc" in objetos:
    for tabla, d in tablas_ssc.items():
        for col in d.columns:
            perfil_vacios.append({
                "tabla": tabla,
                "columna": col,
                "filas": len(d),
                "vacios": int(d[col].isna().sum()),
                "porcentaje_vacio": round(d[col].isna().mean()*100,2)
            })
else:
    # Elegir la tabla más completa disponible.
    d = c5 if "c5" in globals() else (datos if "datos" in globals() else (cdmx if "cdmx" in globals() else datos_limpios))
    for col in d.columns:
        perfil_vacios.append({
            "tabla": "base",
            "columna": col,
            "filas": len(d),
            "vacios": int(d[col].isna().sum()),
            "porcentaje_vacio": round(d[col].isna().mean()*100,2)
        })

perfil_vacios = pd.DataFrame(perfil_vacios)
display(perfil_vacios.sort_values(["tabla","porcentaje_vacio"],ascending=[True,False]))


## Auditoría 2 — Formatos diferentes o no interpretables

Se resumen los valores que no pudieron interpretarse correctamente o quedaron fuera de un rango esperado. La fila se conserva y el problema queda señalado mediante columnas de control.

Las columnas de control creadas durante la limpieza (`_...invalida`, `_...fuera_rango`, etc.) permiten contar problemas de formato sin borrar el valor original.


In [ ]:
problemas_formato = []
bases_revision = tablas_ssc if "tablas_ssc" in globals() else {"base": (c5 if "c5" in globals() else datos)}

for tabla,d in bases_revision.items():
    controles = [c for c in d.columns if c.startswith("_") and any(
        palabra in c for palabra in ["invalida","invalido","fuera_rango","negativo","no_reconocida"]
    )]
    for c in controles:
        problemas_formato.append({
            "tabla": tabla,
            "control": c,
            "registros_afectados": int(d[c].fillna(False).astype(bool).sum())
        })

problemas_formato = pd.DataFrame(problemas_formato)
display(problemas_formato)


## Auditoría 3 — Duplicados exactos y valores repetidos

Se buscan únicamente copias exactas de las filas. Se conserva una primera aparición y, cuando corresponde, las copias exactas adicionales se separan para evitar doble conteo. Una coincidencia parcial no se elimina.

Se distinguen dos situaciones: una copia exacta de toda la fila y un identificador/folio que aparece varias veces. Solo la primera situación puede separarse automáticamente.


In [ ]:
resumen_repetidos = []
bases_revision = tablas_ssc if "tablas_ssc" in globals() else {"base": (c5 if "c5" in globals() else datos)}

for tabla,d in bases_revision.items():
    duplicados = int(d["_duplicado_exacto"].sum()) if "_duplicado_exacto" in d else int(d.duplicated().sum())
    fila = {
        "tabla": tabla,
        "filas_totales": len(d),
        "copias_exactas_adicionales": duplicados,
        "filas_conservables_si_solo_separan_exactos": len(d)-duplicados
    }
    if "folio" in d.columns:
        fila["folios_distintos_repetidos"] = int(
            d.loc[d["folio"].notna() & d.duplicated("folio",keep=False),"folio"].nunique()
        )
        fila["filas_con_folio_repetido"] = int(
            (d["folio"].notna() & d.duplicated("folio",keep=False)).sum()
        )
    resumen_repetidos.append(fila)

resumen_repetidos = pd.DataFrame(resumen_repetidos)
display(resumen_repetidos)


## Auditoría 4 — Qué datos se conservan y cuáles se separan

Se contabilizan explícitamente las filas que permanecen en la base y las que se separan. La conciliación comprueba que ninguna fila desaparezca sin una razón documentada.

La conciliación debe cumplir: **filas originales = filas conservadas + copias exactas adicionales separadas**. Los faltantes, formatos problemáticos y folios repetidos no se eliminan automáticamente.


In [ ]:
conciliacion_auditable = []
bases_revision = tablas_ssc if "tablas_ssc" in globals() else {"base": (c5 if "c5" in globals() else datos)}

for tabla,d in bases_revision.items():
    mascara_dup = d["_duplicado_exacto"] if "_duplicado_exacto" in d else d.duplicated(keep="first")
    conciliacion_auditable.append({
        "tabla": tabla,
        "filas_originales": len(d),
        "filas_conservadas": int((~mascara_dup).sum()),
        "filas_separadas": int(mascara_dup.sum()),
        "motivo_separacion": "Copia exacta adicional",
        "faltantes_eliminados": 0,
        "folios_repetidos_eliminados": 0
    })

conciliacion_auditable = pd.DataFrame(conciliacion_auditable)
conciliacion_auditable["comprobacion"] = (
    conciliacion_auditable["filas_originales"] ==
    conciliacion_auditable["filas_conservadas"] + conciliacion_auditable["filas_separadas"]
)
display(conciliacion_auditable)
assert conciliacion_auditable["comprobacion"].all()


## Bitácora de decisiones

La bitácora registra cada decisión metodológica: la acción realizada, los datos que se conservan, los que se separan y la justificación. Esto permite explicar y reproducir la limpieza.

Esta bitácora resume las decisiones metodológicas. Los conteos de registros afectados se obtienen de las auditorías anteriores.


In [ ]:
bitacora_decisiones = pd.DataFrame([
    {
        "fuente":"C5",
        "decision":"Conservar valores originales",
        "accion":"No sobrescribir columnas crudas",
        "datos_que_se_conservan":"Todos los valores originales",
        "datos_que_se_separan":"Ninguno por esta regla",
        "justificacion":"Mantener trazabilidad y permitir comparar antes y después."
    },
    {
        "fuente":"C5",
        "decision":"Crear columnas limpias",
        "accion":"Agregar columnas derivadas sin reemplazar las originales",
        "datos_que_se_conservan":"Original + versión limpia",
        "datos_que_se_separan":"Ninguno",
        "justificacion":"Evitar pérdida de información."
    },
    {
        "fuente":"C5",
        "decision":"Mantener faltantes",
        "accion":"No imputar valores sin evidencia",
        "datos_que_se_conservan":"Filas con datos faltantes",
        "datos_que_se_separan":"Ninguno solo por estar vacío",
        "justificacion":"Vacío no equivale a cero ni a una categoría conocida."
    },
    {
        "fuente":"C5",
        "decision":"Mantener identificadores repetidos",
        "accion":"Marcar para revisión sin eliminar",
        "datos_que_se_conservan":"Registros con folio/identificador repetido",
        "datos_que_se_separan":"Ninguno solo por repetición",
        "justificacion":"La repetición del identificador no demuestra que las filas sean copias."
    },
    {
        "fuente":"C5",
        "decision":"Separar duplicados exactos",
        "accion":"Conservar primera aparición y separar copias exactas adicionales",
        "datos_que_se_conservan":"Primera copia y todas las filas no idénticas",
        "datos_que_se_separan":"Copias exactas adicionales",
        "justificacion":"Evitar doble conteo sin eliminar registros distintos."
    }
])
display(bitacora_decisiones)


## Exportar evidencias de auditoría como CSV

Los resultados de la limpieza y las evidencias de auditoría se guardan como CSV independientes. Así se conserva por separado la base preparada, los controles de calidad, la conciliación y la bitácora.

Estos archivos son pequeños y pueden descargarse individualmente junto con las bases limpias.


In [ ]:
# Usa la carpeta SALIDA que ya fue creada por el notebook.
perfil_vacios.to_csv(SALIDA/"perfil_vacios.csv",index=False,encoding="utf-8-sig")
problemas_formato.to_csv(SALIDA/"problemas_formato.csv",index=False,encoding="utf-8-sig")
resumen_repetidos.to_csv(SALIDA/"resumen_repetidos.csv",index=False,encoding="utf-8-sig")
conciliacion_auditable.to_csv(SALIDA/"conciliacion_auditable.csv",index=False,encoding="utf-8-sig")
bitacora_decisiones.to_csv(SALIDA/"bitacora_decisiones.csv",index=False,encoding="utf-8-sig")

print("Evidencias de auditoría generadas.")


# Resumen para documentar la limpieza

La limpieza se diseñó para conservar la trazabilidad de la información. Los valores originales permanecen disponibles y las transformaciones se almacenan en columnas nuevas. Los valores faltantes no se sustituyen por ceros, categorías ni otros valores inventados. Los formatos no interpretables se marcan mediante columnas de control y el registro se conserva para que el problema pueda revisarse.

Los valores repetidos se analizan según su significado. Un folio o identificador repetido no se considera automáticamente un duplicado, porque puede representar registros distintos relacionados con un mismo hecho. Por esta razón, los folios repetidos se contabilizan y marcan, pero no se eliminan. La separación automática se limita a copias exactas adicionales de una fila, conservando siempre una primera aparición.

Para demostrar qué ocurrió durante la preparación se generan tablas de auditoría con el número de vacíos por columna, problemas de formato, duplicados exactos, identificadores repetidos y una conciliación entre filas originales, conservadas y separadas. Finalmente, cada decisión metodológica se registra en una bitácora y todos estos productos pueden descargarse individualmente en formato CSV.


## Descripción final del proceso — C5

En esta fuente se mantuvo como principio principal la **trazabilidad**. Los datos originales no se sobrescriben y las versiones preparadas se almacenan en columnas nuevas. Los valores faltantes se conservan como faltantes y los formatos diferentes o no interpretables se identifican mediante controles específicos.

También se distingue entre una **copia exacta** y un **valor repetido**. Los folios o identificadores que aparecen varias veces no se eliminan solamente por repetirse. Cuando existe una copia exacta adicional, se documenta y puede separarse conservando una primera aparición. Finalmente, la conciliación, los perfiles de vacíos y formatos, el resumen de repetidos y la bitácora permiten demostrar qué información se conservó, qué se separó y por qué.
